In [10]:
import pandas as pd
import spotipy

In [11]:
df = pd.read_csv(
    "../data/processed/spotify/lastfm_recenttracks.csv",
    encoding="utf-8"
)

In [12]:
df = df[["track_name", "artist_name"]]
df.head()

,track_name,artist_name
0,Professional,The Weeknd
1,Vampiro,Matuê
2,Heartbeat,Childish Gambino
3,When I’m Home,James Blake
4,Both (feat. Drake),Gucci Mane


In [13]:
df = df.drop_duplicates(["track_name", "artist_name"])
df = df.head(10)

In [14]:
df["spotify_id"] = df.apply(
    lambda row: get_track_id(row["track_name"], row["artist_name"], CLIENT_ID, CLIENT_SECRET), axis=1
)

df["isrc"] = df.apply(
    lambda row: get_track_isrc(row["track_name"], row["artist_name"], CLIENT_ID, CLIENT_SECRET), axis=1
)

In [ ]:
def get_features_safe(row):
    try:
        return get_track_features(row["isrc"])
    except Exception as e:
        print(
            f"Erro: {row['artist_name']} - "
            f"{row['track_name']} - "
            f"{row['isrc']} | {e}"
        )
        return None


df["features"] = df.apply(
    get_features_safe,
    axis=1
)

Erro: The Weeknd - Professional - USUM71310341 | 500 Server Error: Internal Server Error for url: https://melodata1.p.rapidapi.com/tracks/USUM71310341/features
Erro: Matuê - Vampiro - BXW922100034 | 500 Server Error: Internal Server Error for url: https://melodata1.p.rapidapi.com/tracks/BXW922100034/features
Erro: Childish Gambino - Heartbeat - USYAH1100351 | 500 Server Error: Internal Server Error for url: https://melodata1.p.rapidapi.com/tracks/USYAH1100351/features
Erro: James Blake - When I’m Home - USQ4E2600373 | 500 Server Error: Internal Server Error for url: https://melodata1.p.rapidapi.com/tracks/USQ4E2600373/features
Erro: Gucci Mane - Both (feat. Drake) - USAT21603373 | 500 Server Error: Internal Server Error for url: https://melodata1.p.rapidapi.com/tracks/USAT21603373/features
Erro: Jhené Aiko - Stay Ready (What a Life) - USUM71312344 | 500 Server Error: Internal Server Error for url: https://melodata1.p.rapidapi.com/tracks/USUM71312344/features
Erro: Frank Ocean - Novacan

In [ ]:
df.to_csv("parte_1.csv", index=False, encoding="utf-8")

In [16]:
import requests

def search_recco(track_name, artist_name):
    url = "https://api.reccobeats.com/v1/track/search"

    params = {
        "searchText": track_name,
        "artist": artist_name
    }

    response = requests.get(url, params=params)
    response.raise_for_status()

    return response.json()

def search_recco_safe(row):
    try:
        return search_recco(row["track_name"], row["artist_name"])
    except Exception as e:
        print(
            f"Erro: {row['artist_name']} - "
            f"{row['track_name']} - "
            f"{row['isrc']} | {e}"
        )
        return None

df["recco_result"] = df.apply(
    lambda row: search_recco_safe(row),
    axis=1
)

df.head()

,track_name,artist_name,spotify_id,isrc,features,recco_result
0,Professional,The Weeknd,5ZicFGBDAi9J2YCVesboUp,USUM71310341,None,{'content': [{'id': 'c9ea3a2d-dc6a-4462-9aa6-5...
1,Vampiro,Matuê,6bTdZ7xfKp3NqqADJ8HLyj,BXW922100034,None,{'content': [{'id': '8492861c-06b0-44aa-a991-e...
2,Heartbeat,Childish Gambino,5AGQSF0ytihJyt96K5vW9d,USYAH1100351,None,{'content': [{'id': '18f5a77a-bdf4-4a11-a3f4-3...
3,When I’m Home,James Blake,5nigMKgXqeNvaiqEBWwo9s,USQ4E2600373,None,"{'content': [], 'page': 0, 'size': 25, 'totalE..."
4,Both (feat. Drake),Gucci Mane,5tFep7dXGd7vEJ668wTPux,USAT21603373,None,{'content': [{'id': 'f8186f9d-cdac-44f0-86bf-a...


In [17]:
df.iloc[1]["recco_result"]

{'content': [{'id': '8492861c-06b0-44aa-a991-ebeb3d1ef53e',
   'trackTitle': 'Vampiro',
   'artists': [{'id': '3cba4e20-f96a-4aa4-a82f-0afead857a2d',
     'name': 'Matuê',
     'href': 'https://open.spotify.com/artist/5nP8x4uEFjAAmDzwOEc9b8'},
    {'id': '188e6f5d-b392-486b-9b4c-d97cdacc50a3',
     'name': 'WIU',
     'href': 'https://open.spotify.com/artist/3MrDVzg7ZXaYMyQmbDInr7'},
    {'id': '2d6069a8-0f61-4ac0-814e-44c5e45c0fb4',
     'name': 'Teto',
     'href': 'https://open.spotify.com/artist/68YeXpLt3jB7JHQS5ZjMGo'}],
   'durationMs': 250434,
   'isrc': 'BXW922100034',
   'ean': None,
   'upc': None,
   'href': 'https://open.spotify.com/track/6bTdZ7xfKp3NqqADJ8HLyj',
   'availableCountries': 'AR,AU,AT,BE,BO,BR,BG,CA,CL,CO,CR,CY,CZ,DK,DO,DE,EC,EE,SV,FI,FR,GR,GT,HN,HK,HU,IS,IE,IT,LV,LT,LU,MY,MT,MX,NL,NZ,NI,NO,PA,PY,PE,PH,PL,PT,SG,SK,ES,SE,CH,TW,TR,UY,US,GB,AD,LI,MC,ID,JP,TH,VN,RO,IL,ZA,SA,AE,BH,QA,OM,KW,EG,MA,DZ,TN,LB,JO,PS,IN,BY,KZ,MD,UA,AL,BA,HR,ME,MK,RS,SI,KR,BD,PK,LK,GH,KE,NG

In [18]:
df["recco_id"] = df["recco_result"].apply(
    lambda x: x["content"][0]["id"] if x["content"] else None
)

df.drop(["recco_result"], axis=1, inplace=True)
df

,track_name,artist_name,spotify_id,isrc,features,recco_id
0,Professional,The Weeknd,5ZicFGBDAi9J2YCVesboUp,USUM71310341,None,c9ea3a2d-dc6a-4462-9aa6-56ff93f3b60b
1,Vampiro,Matuê,6bTdZ7xfKp3NqqADJ8HLyj,BXW922100034,None,8492861c-06b0-44aa-a991-ebeb3d1ef53e
2,Heartbeat,Childish Gambino,5AGQSF0ytihJyt96K5vW9d,USYAH1100351,None,18f5a77a-bdf4-4a11-a3f4-3e4b570cb814
3,When I’m Home,James Blake,5nigMKgXqeNvaiqEBWwo9s,USQ4E2600373,None,None
4,Both (feat. Drake),Gucci Mane,5tFep7dXGd7vEJ668wTPux,USAT21603373,None,f8186f9d-cdac-44f0-86bf-aa18ca19a753
5,Stay Ready (What a Life),Jhené Aiko,5nkUIVKqOqdpB6ApKgEMkv,USUM71312344,None,f1920772-e80a-488b-833c-04b4f2956535
6,Jungle,Drake,7JXZq0JgG2zTrSOAgY8VMC,USCM51500037,"{'data': {'isrc': 'USCM51500037', 'title': 'Ju...",f30a1ce4-7adb-4ca7-bdc7-9f70110f90d1
8,Novacane,Frank Ocean,4osgfFTICMkcGbbigdsa53,USUM71107257,None,65c51efa-45e3-415c-b770-c7f4a2f45866
9,The Zone,The Weeknd,53qYItjefG5SUf62428dIw,USUM72111952,None,e52ef438-7cf1-47ac-bb60-f800b52ea3d6
10,Lose You,Drake,2Na0z2gfN67Rzf0vp74Wi3,USCM51700071,None,None


In [19]:
import requests


def get_recco_audio_features(track_id):
    url = f"https://api.reccobeats.com/v1/track/{track_id}/audio-features"

    headers = {
        "Accept": "application/json"
    }

    response = requests.get(
        url,
        headers=headers
    )

    response.raise_for_status()

    return response.json()

In [20]:
df["features_recco"] = df["recco_id"].apply(
    lambda x: get_recco_audio_features(x) if x else None)

df

,track_name,artist_name,spotify_id,isrc,features,recco_id,features_recco
0,Professional,The Weeknd,5ZicFGBDAi9J2YCVesboUp,USUM71310341,None,c9ea3a2d-dc6a-4462-9aa6-56ff93f3b60b,"{'id': 'c9ea3a2d-dc6a-4462-9aa6-56ff93f3b60b',..."
1,Vampiro,Matuê,6bTdZ7xfKp3NqqADJ8HLyj,BXW922100034,None,8492861c-06b0-44aa-a991-ebeb3d1ef53e,"{'id': '8492861c-06b0-44aa-a991-ebeb3d1ef53e',..."
2,Heartbeat,Childish Gambino,5AGQSF0ytihJyt96K5vW9d,USYAH1100351,None,18f5a77a-bdf4-4a11-a3f4-3e4b570cb814,"{'id': '18f5a77a-bdf4-4a11-a3f4-3e4b570cb814',..."
3,When I’m Home,James Blake,5nigMKgXqeNvaiqEBWwo9s,USQ4E2600373,None,None,None
4,Both (feat. Drake),Gucci Mane,5tFep7dXGd7vEJ668wTPux,USAT21603373,None,f8186f9d-cdac-44f0-86bf-aa18ca19a753,"{'id': 'f8186f9d-cdac-44f0-86bf-aa18ca19a753',..."
5,Stay Ready (What a Life),Jhené Aiko,5nkUIVKqOqdpB6ApKgEMkv,USUM71312344,None,f1920772-e80a-488b-833c-04b4f2956535,"{'id': 'f1920772-e80a-488b-833c-04b4f2956535',..."
6,Jungle,Drake,7JXZq0JgG2zTrSOAgY8VMC,USCM51500037,"{'data': {'isrc': 'USCM51500037', 'title': 'Ju...",f30a1ce4-7adb-4ca7-bdc7-9f70110f90d1,"{'id': 'f30a1ce4-7adb-4ca7-bdc7-9f70110f90d1',..."
8,Novacane,Frank Ocean,4osgfFTICMkcGbbigdsa53,USUM71107257,None,65c51efa-45e3-415c-b770-c7f4a2f45866,"{'id': '65c51efa-45e3-415c-b770-c7f4a2f45866',..."
9,The Zone,The Weeknd,53qYItjefG5SUf62428dIw,USUM72111952,None,e52ef438-7cf1-47ac-bb60-f800b52ea3d6,"{'id': 'e52ef438-7cf1-47ac-bb60-f800b52ea3d6',..."
10,Lose You,Drake,2Na0z2gfN67Rzf0vp74Wi3,USCM51700071,None,None,None


In [21]:
df_features = df.copy()[["track_name", "features", "features_recco"]]

In [22]:
df_features.iloc[0]["features_recco"]

{'id': 'c9ea3a2d-dc6a-4462-9aa6-56ff93f3b60b',
 'href': 'https://open.spotify.com/track/5ZicFGBDAi9J2YCVesboUp',
 'isrc': 'USUM71310341',
 'acousticness': 0.163,
 'danceability': 0.405,
 'energy': 0.619,
 'instrumentalness': 0.000322,
 'key': 11,
 'liveness': 0.0788,
 'loudness': -8.92,
 'mode': 0,
 'speechiness': 0.0618,
 'tempo': 120.087,
 'valence': 0.234}

In [23]:
features = pd.json_normalize(df["features_recco"])

df = pd.concat(
    [df.drop(columns=["features_recco"]), features],
    axis=1
)

df

,track_name,artist_name,spotify_id,isrc,features,recco_id,id,href,isrc,acousticness,danceability,energy,instrumentalness,key,liveness,loudness,mode,speechiness,tempo,valence
0,Professional,The Weeknd,5ZicFGBDAi9J2YCVesboUp,USUM71310341,None,c9ea3a2d-dc6a-4462-9aa6-56ff93f3b60b,c9ea3a2d-dc6a-4462-9aa6-56ff93f3b60b,https://open.spotify.com/track/5ZicFGBDAi9J2YC...,USUM71310341,0.16300,0.405,0.619,0.000322,11.0,0.0788,-8.920,0.0,0.0618,120.087,0.234
1,Vampiro,Matuê,6bTdZ7xfKp3NqqADJ8HLyj,BXW922100034,None,8492861c-06b0-44aa-a991-ebeb3d1ef53e,8492861c-06b0-44aa-a991-ebeb3d1ef53e,https://open.spotify.com/track/6bTdZ7xfKp3NqqA...,BXW922100034,0.01360,0.782,0.643,0.000000,8.0,0.0654,-4.956,0.0,0.0438,114.994,0.627
2,Heartbeat,Childish Gambino,5AGQSF0ytihJyt96K5vW9d,USYAH1100351,None,18f5a77a-bdf4-4a11-a3f4-3e4b570cb814,18f5a77a-bdf4-4a11-a3f4-3e4b570cb814,https://open.spotify.com/track/5AGQSF0ytihJyt9...,USYAH1100351,0.00373,0.800,0.545,0.000511,1.0,0.0445,-7.002,0.0,0.1270,119.959,0.291
3,When I’m Home,James Blake,5nigMKgXqeNvaiqEBWwo9s,USQ4E2600373,None,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Both (feat. Drake),Gucci Mane,5tFep7dXGd7vEJ668wTPux,USAT21603373,None,f8186f9d-cdac-44f0-86bf-aa18ca19a753,f8186f9d-cdac-44f0-86bf-aa18ca19a753,https://open.spotify.com/track/0iQXqF8CYMI7mLN...,USAT21603373,0.11900,0.849,0.405,0.000118,7.0,0.0707,-7.509,0.0,0.2260,139.976,0.344
5,Stay Ready (What a Life),Jhené Aiko,5nkUIVKqOqdpB6ApKgEMkv,USUM71312344,None,f1920772-e80a-488b-833c-04b4f2956535,f1920772-e80a-488b-833c-04b4f2956535,https://open.spotify.com/track/5nkUIVKqOqdpB6A...,USUM71312344,0.45500,0.347,0.493,0.003660,8.0,0.1270,-11.548,0.0,0.2900,82.887,0.308
6,Jungle,Drake,7JXZq0JgG2zTrSOAgY8VMC,USCM51500037,"{'data': {'isrc': 'USCM51500037', 'title': 'Ju...",f30a1ce4-7adb-4ca7-bdc7-9f70110f90d1,f30a1ce4-7adb-4ca7-bdc7-9f70110f90d1,https://open.spotify.com/track/0mnABjjVAEmnSso...,USCM51500038,0.61800,0.685,0.226,0.000229,7.0,0.1070,-8.694,1.0,0.0466,99.786,0.418
8,Novacane,Frank Ocean,4osgfFTICMkcGbbigdsa53,USUM71107257,None,65c51efa-45e3-415c-b770-c7f4a2f45866,e52ef438-7cf1-47ac-bb60-f800b52ea3d6,https://open.spotify.com/track/53qYItjefG5SUf6...,USUM72111952,0.12900,0.654,0.383,0.000755,3.0,0.1030,-9.875,0.0,0.0363,139.950,0.268
9,The Zone,The Weeknd,53qYItjefG5SUf62428dIw,USUM72111952,None,e52ef438-7cf1-47ac-bb60-f800b52ea3d6,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
10,Lose You,Drake,2Na0z2gfN67Rzf0vp74Wi3,USCM51700071,None,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [24]:
df.to_csv("../data/processed/spotify/enriched_data.csv", index=False, encoding="utf-8")

In [1]:
import spotipy
from spotipy.oauth2 import SpotifyClientCredentials


def get_artist_data(artist_name, client_id, client_secret):
    sp = spotipy.Spotify(
        auth_manager=SpotifyClientCredentials(
            client_id=client_id,
            client_secret=client_secret
        )
    )

    query = f"artist:{artist_name}"

    results = sp.search(
        q=query,
        type="artist",
        limit=1
    )

    artists = results["artists"]["items"]

    if not artists:
        return None

    artist = artists[0]

    return {
        "artist_id": artist["id"],
        "artist_name": artist["name"],
        "artist_url": artist["external_urls"].get("spotify"),
        "followers": artist["followers"]["total"],
        "popularity": artist["popularity"],
        "genres": artist["genres"],
        "artist_uri": artist["uri"]
    }

In [3]:
CLIENT_ID = "d61d466b88404563a016f197e8572c5e"
CLIENT_SECRET = "760825fb70dc4d449ad30950e90e904d"

artist_data = get_artist_data(
    "Matue",
    CLIENT_ID,
    CLIENT_SECRET
)

artist_data

{'artist_id': '5nP8x4uEFjAAmDzwOEc9b8',
 'artist_name': 'Matuê',
 'artist_url': 'https://open.spotify.com/artist/5nP8x4uEFjAAmDzwOEc9b8',
 'followers': 15615733,
 'popularity': 79,
 'genres': ['brazilian trap', 'trap', 'brazilian hip hop', 'trap funk'],
 'artist_uri': 'spotify:artist:5nP8x4uEFjAAmDzwOEc9b8'}

In [8]:
import pandas as pd

df_artist = pd.read_csv("../data/processed/spotify/new_artists.csv")
df_artist = df_artist.head(10)

In [10]:
df_artist["spotify_data"] = df_artist["artist_name"].apply(
    lambda x: get_artist_data(x, CLIENT_ID, CLIENT_SECRET)
)

df_artist

,artist_mbid,artist_name,spotify_data
0,NaN,Adele,"{'artist_id': '3XPI2Rik4spEoO1VW3fHk7', 'artis..."
1,875203e1-8e58-4b86-8dcb-7190faf411c5,J. Cole,"{'artist_id': '6l3HvQ5sa6mXTsMTB19rO5', 'artis..."
2,NaN,Drake,"{'artist_id': '3TVXtAsR1Inumwj472S9r4', 'artis..."
3,29e686ca-134e-46d5-8a5b-7193f618ff35,Sonder,"{'artist_id': '2ICR2m4hOBPhaYiZB3rnLW', 'artis..."
4,b7539c32-53e7-4908-bda3-81449c367da6,Lana Del Rey,"{'artist_id': '00FQb4jTyendYWaN8pK0wa', 'artis..."
5,NaN,Pitbull,"{'artist_id': '0TnOYISbd1XYRBk9myaseg', 'artis..."
6,c8b03190-306c-4120-bb0b-6f2ebfc06ea9,The Weeknd,"{'artist_id': '1Xyo4u8uXC1ZmMpatF05PJ', 'artis..."
7,285e1285-aecc-478f-a8da-0c8cf69e1217,Miguel,"{'artist_id': '360IAlyVv4PCEVjgyMZrxK', 'artis..."
8,b95ce3ff-3d05-4e87-9e01-c97b66af13d4,Eminem,"{'artist_id': '7dGJo4pcD2V6oG8kP0tJRR', 'artis..."
9,e0140a67-e4d1-4f13-8a01-364355bee46e,Justin Bieber,"{'artist_id': '1uNFoZAHBGtllmzznpCI3s', 'artis..."


In [11]:
artist_data_spotify = pd.json_normalize(df_artist["spotify_data"])

df_artist = pd.concat(
    [df_artist.drop(columns=["spotify_data"]), artist_data_spotify],
    axis=1
)

df_artist

,artist_mbid,artist_name,artist_id,artist_name,artist_url,followers,popularity,genres,artist_uri
0,NaN,Adele,3XPI2Rik4spEoO1VW3fHk7,Artistnameleon,https://open.spotify.com/artist/3XPI2Rik4spEoO...,1488,16,[sexy drill],spotify:artist:3XPI2Rik4spEoO1VW3fHk7
1,875203e1-8e58-4b86-8dcb-7190faf411c5,J. Cole,6l3HvQ5sa6mXTsMTB19rO5,J. Cole,https://open.spotify.com/artist/6l3HvQ5sa6mXTs...,28840542,87,[rap],spotify:artist:6l3HvQ5sa6mXTsMTB19rO5
2,NaN,Drake,3TVXtAsR1Inumwj472S9r4,Drake,https://open.spotify.com/artist/3TVXtAsR1Inumw...,115450362,100,[rap],spotify:artist:3TVXtAsR1Inumwj472S9r4
3,29e686ca-134e-46d5-8a5b-7193f618ff35,Sonder,2ICR2m4hOBPhaYiZB3rnLW,Sonder,https://open.spotify.com/artist/2ICR2m4hOBPhaY...,2477612,73,"[r&b, dark r&b]",spotify:artist:2ICR2m4hOBPhaYiZB3rnLW
4,b7539c32-53e7-4908-bda3-81449c367da6,Lana Del Rey,00FQb4jTyendYWaN8pK0wa,Lana Del Rey,https://open.spotify.com/artist/00FQb4jTyendYW...,59822313,91,[],spotify:artist:00FQb4jTyendYWaN8pK0wa
5,NaN,Pitbull,0TnOYISbd1XYRBk9myaseg,Pitbull,https://open.spotify.com/artist/0TnOYISbd1XYRB...,13267096,89,[],spotify:artist:0TnOYISbd1XYRBk9myaseg
6,c8b03190-306c-4120-bb0b-6f2ebfc06ea9,The Weeknd,1Xyo4u8uXC1ZmMpatF05PJ,The Weeknd,https://open.spotify.com/artist/1Xyo4u8uXC1ZmM...,126420080,96,[],spotify:artist:1Xyo4u8uXC1ZmMpatF05PJ
7,285e1285-aecc-478f-a8da-0c8cf69e1217,Miguel,360IAlyVv4PCEVjgyMZrxK,Miguel,https://open.spotify.com/artist/360IAlyVv4PCEV...,6952247,83,[r&b],spotify:artist:360IAlyVv4PCEVjgyMZrxK
8,b95ce3ff-3d05-4e87-9e01-c97b66af13d4,Eminem,7dGJo4pcD2V6oG8kP0tJRR,Eminem,https://open.spotify.com/artist/7dGJo4pcD2V6oG...,111683015,91,"[rap, hip hop]",spotify:artist:7dGJo4pcD2V6oG8kP0tJRR
9,e0140a67-e4d1-4f13-8a01-364355bee46e,Justin Bieber,1uNFoZAHBGtllmzznpCI3s,Justin Bieber,https://open.spotify.com/artist/1uNFoZAHBGtllm...,95114896,96,[],spotify:artist:1uNFoZAHBGtllmzznpCI3s


In [19]:
def get_album_data(album_name, client_id, client_secret):

    sp = spotipy.Spotify(
        auth_manager=SpotifyClientCredentials(
            client_id=client_id,
            client_secret=client_secret
        )
    )

    query = f"album:{album_name}"

    results = sp.search(
        q=query,
        type="album",
        limit=1
    )

    albums = results["albums"]["items"]

    if not albums:
        return None

    album = albums[0]

    return {
        "album_id": album["id"],
        #"album_name": album["name"],
        "album_type": album["album_type"],
        "total_tracks": album["total_tracks"],
        "release_date": album["release_date"],
        "release_date_precision": album["release_date_precision"],
        "artist_id": album["artists"][0]["id"] if album["artists"] else None,
        #"artist_name": album["artists"][0]["name"] if album["artists"] else None,
        "album_url": album["external_urls"].get("spotify"),
        "album_uri": album["uri"],
        #"image_url": album["images"][0]["url"] if album["images"] else None
    }

In [20]:
df_album = pd.read_csv("../data/processed/spotify/new_albums.csv")
df_album = df_album.head(10)
df_album

,album_mbid,album_name
0,29f224ac-498d-4dcb-9a07-22b1293288ae,21
1,32ef7c3f-96bb-44a3-aa4d-7aa8c9640cf9,2014 Forest Hills Drive
2,NaN,Take Care (Deluxe)
3,16a0fd7d-307d-4721-9c2b-ddb0d1d847d8,What You Heard
4,NaN,Snow On Tha Bluff
5,620e4932-cae3-49d0-9d87-a2685bd4c9c0,4 Your Eyez Only
6,4e102cda-b8f6-4242-b4ae-ce2d16a5eac7,Say Yes to Heaven
7,NaN,Born to Die - The Paradise Edition
8,NaN,Ultraviolence (Deluxe)
9,0c6272c4-f50a-4f80-a302-497038a7f5a0,Young and Beautiful


In [21]:
df_album["spotify_data"] = df_album.apply(
    lambda row: get_album_data(
        row["album_name"],
        # row["artist_name"],
        client_id= CLIENT_ID,
        client_secret=CLIENT_SECRET
    ),
    axis=1
)
df_album

,album_mbid,album_name,spotify_data
0,29f224ac-498d-4dcb-9a07-22b1293288ae,21,"{'album_id': '0Lg1uZvI312TPqxNWShFXL', 'album_..."
1,32ef7c3f-96bb-44a3-aa4d-7aa8c9640cf9,2014 Forest Hills Drive,"{'album_id': '0UMMIkurRUmkruZ3KGBLtG', 'album_..."
2,NaN,Take Care (Deluxe),"{'album_id': '6X1x82kppWZmDzlXXK3y3q', 'album_..."
3,16a0fd7d-307d-4721-9c2b-ddb0d1d847d8,What You Heard,"{'album_id': '10WCcQKzXZot04kzENu62Z', 'album_..."
4,NaN,Snow On Tha Bluff,"{'album_id': '0MXpE6m0mEK0r3TuERb9Yd', 'album_..."
5,620e4932-cae3-49d0-9d87-a2685bd4c9c0,4 Your Eyez Only,"{'album_id': '3CCnGldVQ90c26aFATC1PW', 'album_..."
6,4e102cda-b8f6-4242-b4ae-ce2d16a5eac7,Say Yes to Heaven,"{'album_id': '6jVg0POvGYH1Pt6lISl3ok', 'album_..."
7,NaN,Born to Die - The Paradise Edition,"{'album_id': '5VoeRuTrGhTbKelUfwymwu', 'album_..."
8,NaN,Ultraviolence (Deluxe),"{'album_id': '1ORxRsK3MrSLvh7VQTF01F', 'album_..."
9,0c6272c4-f50a-4f80-a302-497038a7f5a0,Young and Beautiful,"{'album_id': '1D92WOHWUI2AGQCCdplcXL', 'album_..."


In [22]:
album_data_spotify = pd.json_normalize(df_album["spotify_data"])

df_album = pd.concat(
    [df_album.drop(columns=["spotify_data"]), album_data_spotify],
    axis=1
)

df_album

,album_mbid,album_name,album_id,album_type,total_tracks,release_date,release_date_precision,artist_id,album_url,album_uri
0,29f224ac-498d-4dcb-9a07-22b1293288ae,21,0Lg1uZvI312TPqxNWShFXL,album,11,2011-01-24,day,4dpARuHxo51G3z768sgnrY,https://open.spotify.com/album/0Lg1uZvI312TPqx...,spotify:album:0Lg1uZvI312TPqxNWShFXL
1,32ef7c3f-96bb-44a3-aa4d-7aa8c9640cf9,2014 Forest Hills Drive,0UMMIkurRUmkruZ3KGBLtG,album,13,2014-12-09,day,6l3HvQ5sa6mXTsMTB19rO5,https://open.spotify.com/album/0UMMIkurRUmkruZ...,spotify:album:0UMMIkurRUmkruZ3KGBLtG
2,NaN,Take Care (Deluxe),6X1x82kppWZmDzlXXK3y3q,album,19,2011-11-15,day,3TVXtAsR1Inumwj472S9r4,https://open.spotify.com/album/6X1x82kppWZmDzl...,spotify:album:6X1x82kppWZmDzlXXK3y3q
3,16a0fd7d-307d-4721-9c2b-ddb0d1d847d8,What You Heard,10WCcQKzXZot04kzENu62Z,single,1,2019-02-22,day,2ICR2m4hOBPhaYiZB3rnLW,https://open.spotify.com/album/10WCcQKzXZot04k...,spotify:album:10WCcQKzXZot04kzENu62Z
4,NaN,Snow On Tha Bluff,0MXpE6m0mEK0r3TuERb9Yd,single,1,2020-06-16,day,6l3HvQ5sa6mXTsMTB19rO5,https://open.spotify.com/album/0MXpE6m0mEK0r3T...,spotify:album:0MXpE6m0mEK0r3TuERb9Yd
5,620e4932-cae3-49d0-9d87-a2685bd4c9c0,4 Your Eyez Only,3CCnGldVQ90c26aFATC1PW,album,10,2016-12-09,day,6l3HvQ5sa6mXTsMTB19rO5,https://open.spotify.com/album/3CCnGldVQ90c26a...,spotify:album:3CCnGldVQ90c26aFATC1PW
6,4e102cda-b8f6-4242-b4ae-ce2d16a5eac7,Say Yes to Heaven,6jVg0POvGYH1Pt6lISl3ok,single,2,2023-05-19,day,00FQb4jTyendYWaN8pK0wa,https://open.spotify.com/album/6jVg0POvGYH1Pt6...,spotify:album:6jVg0POvGYH1Pt6lISl3ok
7,NaN,Born to Die - The Paradise Edition,5VoeRuTrGhTbKelUfwymwu,album,23,2012-01-01,day,00FQb4jTyendYWaN8pK0wa,https://open.spotify.com/album/5VoeRuTrGhTbKel...,spotify:album:5VoeRuTrGhTbKelUfwymwu
8,NaN,Ultraviolence (Deluxe),1ORxRsK3MrSLvh7VQTF01F,album,14,2014-01-01,day,00FQb4jTyendYWaN8pK0wa,https://open.spotify.com/album/1ORxRsK3MrSLvh7...,spotify:album:1ORxRsK3MrSLvh7VQTF01F
9,0c6272c4-f50a-4f80-a302-497038a7f5a0,Young and Beautiful,1D92WOHWUI2AGQCCdplcXL,single,1,2013-01-01,day,00FQb4jTyendYWaN8pK0wa,https://open.spotify.com/album/1D92WOHWUI2AGQC...,spotify:album:1D92WOHWUI2AGQCCdplcXL


In [26]:
def get_track_data(song_name, artist_name, client_id, client_secret):
    query = f"track:{song_name} artist:{artist_name}"

    sp = spotipy.Spotify(
        auth_manager=SpotifyClientCredentials(
            client_id=client_id,
            client_secret=client_secret
        )
    )

    results = sp.search(
        q=query,
        type="track",
        limit=1
    )

    tracks = results["tracks"]["items"]

    if not tracks:
        return None

    track = tracks[0]

    return {
        "track_id": track["id"],
        #"track_name": track["name"],
        #"artist_id": track["artists"][0]["id"] if track["artists"] else None,
        #"artist_name": track["artists"][0]["name"] if track["artists"] else None,
        #"album_id": track["album"]["id"] if track["album"] else None,
        #"album_name": track["album"]["name"] if track["album"] else None,
        #"album_type": track["album"]["album_type"] if track["album"] else None,
        "release_date": track["album"]["release_date"] if track["album"] else None,
        "duration_ms": track["duration_ms"],
        "explicit": track["explicit"],
        "track_number": track["track_number"],
        "disc_number": track["disc_number"],
        "popularity": track["popularity"],
        "track_url": track["external_urls"].get("spotify"),
        "track_uri": track["uri"],
        "preview_url": track["preview_url"]
    }

In [27]:
df_tracks = pd.read_csv("../data/processed/spotify/new_tracks.csv")
df_tracks = df_tracks.head(10)

In [28]:
df_tracks["spotify_data"] = df_tracks.apply(
    lambda row: get_track_data(
        row["track_name"],
        row["artist_name"],
        client_id= CLIENT_ID,
        client_secret=CLIENT_SECRET
    ),
    axis=1
)
df_tracks

,track_mbid,track_name,artist_name,spotify_data
0,0309071e-e8b6-4263-a68d-c4e4f8c4a988,Lovesong,Adele,"{'track_id': '7nm6DlSzzJTH1rk2e6EgJz', 'releas..."
1,f7f85d65-bd9d-4821-9d86-cc0d669bda6c,Apparently,J. Cole,"{'track_id': '5O59s7bCgTFsXDXlWecyQ1', 'releas..."
2,NaN,Cameras / Good Ones Go Interlude - Medley,Drake,"{'track_id': '2FbGlEPAjNhWvrVvlentVq', 'releas..."
3,50acbb51-2ddb-451b-8e6e-acbe1dc9799a,What You Heard,Sonder,"{'track_id': '3a3dQOO19moXPeTt2PomoT', 'releas..."
4,ebb59e72-1549-438d-88e9-c27b697f9e90,Snow On Tha Bluff,J. Cole,"{'track_id': '1oOEkBNp4zWnkD7nWjJdog', 'releas..."
5,64f84b86-2059-4a01-aab0-c7cd8e9aa341,4 Your Eyez Only,J. Cole,"{'track_id': '1vvnYpYEMVB4aq9I6tHIEB', 'releas..."
6,47d724dd-ef88-4337-951e-a42647c21780,She's Mine Pt. 2,J. Cole,"{'track_id': '0HtOJj7Kl74s1Ngf3MWeif', 'releas..."
7,346c038a-1bd5-43f6-9bb5-b6e805e219b2,Foldin Clothes,J. Cole,"{'track_id': '77IAeEz8LEchPN8UNjaTJ2', 'releas..."
8,303a8f69-1196-47e2-82f6-9fcdd09cf09e,Neighbors,J. Cole,"{'track_id': '0utlOiJy2weVl9WTkcEWHy', 'releas..."
9,0d32af4f-5493-4f3b-a2ad-f61fa55698e6,Change,J. Cole,"{'track_id': '3pjUyVbFmM96tYhSaKJwTt', 'releas..."


In [29]:
track_data_spotify = pd.json_normalize(df_tracks["spotify_data"])

df_tracks = pd.concat(
    [df_tracks.drop(columns=["spotify_data"]), track_data_spotify],
    axis=1
)

df_tracks

,track_mbid,track_name,artist_name,track_id,release_date,duration_ms,explicit,track_number,disc_number,popularity,track_url,track_uri,preview_url
0,0309071e-e8b6-4263-a68d-c4e4f8c4a988,Lovesong,Adele,7nm6DlSzzJTH1rk2e6EgJz,2011-01-24,316240,False,10,1,66,https://open.spotify.com/track/7nm6DlSzzJTH1rk...,spotify:track:7nm6DlSzzJTH1rk2e6EgJz,None
1,f7f85d65-bd9d-4821-9d86-cc0d669bda6c,Apparently,J. Cole,5O59s7bCgTFsXDXlWecyQ1,2014-12-09,292865,True,11,1,72,https://open.spotify.com/track/5O59s7bCgTFsXDX...,spotify:track:5O59s7bCgTFsXDXlWecyQ1,None
2,NaN,Cameras / Good Ones Go Interlude - Medley,Drake,2FbGlEPAjNhWvrVvlentVq,2011-11-15,434960,True,12,1,77,https://open.spotify.com/track/2FbGlEPAjNhWvrV...,spotify:track:2FbGlEPAjNhWvrVvlentVq,None
3,50acbb51-2ddb-451b-8e6e-acbe1dc9799a,What You Heard,Sonder,3a3dQOO19moXPeTt2PomoT,2019-02-22,238242,True,1,1,81,https://open.spotify.com/track/3a3dQOO19moXPeT...,spotify:track:3a3dQOO19moXPeTt2PomoT,None
4,ebb59e72-1549-438d-88e9-c27b697f9e90,Snow On Tha Bluff,J. Cole,1oOEkBNp4zWnkD7nWjJdog,2020-06-16,235480,True,1,1,65,https://open.spotify.com/track/1oOEkBNp4zWnkD7...,spotify:track:1oOEkBNp4zWnkD7nWjJdog,None
5,64f84b86-2059-4a01-aab0-c7cd8e9aa341,4 Your Eyez Only,J. Cole,1vvnYpYEMVB4aq9I6tHIEB,2016-12-09,530253,True,10,1,67,https://open.spotify.com/track/1vvnYpYEMVB4aq9...,spotify:track:1vvnYpYEMVB4aq9I6tHIEB,None
6,47d724dd-ef88-4337-951e-a42647c21780,She's Mine Pt. 2,J. Cole,0HtOJj7Kl74s1Ngf3MWeif,2016-12-09,209080,True,5,1,65,https://open.spotify.com/track/0HtOJj7Kl74s1Ng...,spotify:track:0HtOJj7Kl74s1Ngf3MWeif,None
7,346c038a-1bd5-43f6-9bb5-b6e805e219b2,Foldin Clothes,J. Cole,77IAeEz8LEchPN8UNjaTJ2,2016-12-09,316920,True,8,1,58,https://open.spotify.com/track/77IAeEz8LEchPN8...,spotify:track:77IAeEz8LEchPN8UNjaTJ2,None
8,303a8f69-1196-47e2-82f6-9fcdd09cf09e,Neighbors,J. Cole,0utlOiJy2weVl9WTkcEWHy,2016-12-09,216520,True,7,1,73,https://open.spotify.com/track/0utlOiJy2weVl9W...,spotify:track:0utlOiJy2weVl9WTkcEWHy,None
9,0d32af4f-5493-4f3b-a2ad-f61fa55698e6,Change,J. Cole,3pjUyVbFmM96tYhSaKJwTt,2016-12-09,331480,True,6,1,71,https://open.spotify.com/track/3pjUyVbFmM96tYh...,spotify:track:3pjUyVbFmM96tYhSaKJwTt,None


In [2]:
import spotipy
from spotipy.oauth2 import SpotifyClientCredentials


def get_track_isrc(song_name, artist_name, client_id, client_secret):
    sp = spotipy.Spotify(
        auth_manager=SpotifyClientCredentials(
            client_id=client_id,
            client_secret=client_secret
        )
    )

    query = f"track:{song_name} artist:{artist_name}"

    results = sp.search(
        q=query,
        type="track",
        limit=1
    )

    tracks = results["tracks"]["items"]

    if not tracks:
        return None

    return tracks[0]["external_ids"].get("isrc")

In [3]:
def get_track_id(song_name, artist_name, client_id, client_secret):
    sp = spotipy.Spotify(
        auth_manager=SpotifyClientCredentials(
            client_id=client_id,
            client_secret=client_secret
        )
    )

    query = f"track:{song_name} artist:{artist_name}"

    results = sp.search(
        q=query,
        type="track",
        limit=1
    )

    tracks = results["tracks"]["items"]

    if not tracks:
        return None

    return tracks[0]["id"]

In [4]:
CLIENT_ID = "d61d466b88404563a016f197e8572c5e"
CLIENT_SECRET = "760825fb70dc4d449ad30950e90e904d"

id = get_track_id(
    "Blinding Lights",
    "The Weeknd",
    CLIENT_ID,
    CLIENT_SECRET
)

id

'0VjIjW4GlUZAMYd2vXMi3b'

In [5]:
CLIENT_ID = "d61d466b88404563a016f197e8572c5e"
CLIENT_SECRET = "760825fb70dc4d449ad30950e90e904d"

isrc = get_track_isrc(
    "jungle",
    "Drake",
    CLIENT_ID,
    CLIENT_SECRET
)

isrc

'USCM51500037'

In [6]:
import requests


def search_track(song_name, limit=10, offset=0):

    url = "https://melodata1.p.rapidapi.com/tracks/search"

    params = {
        "q": song_name,
        "offset": offset,
        "limit": limit
    }

    headers = {
        "Content-Type": "application/json",
        "x-rapidapi-host": "melodata1.p.rapidapi.com",
        "x-rapidapi-key": "dbcd34ab68mshbdcca93581f9718p1c56b8jsn2fe95019b903"
    }

    response = requests.get(
        url,
        params=params,
        headers=headers
    )

    response.raise_for_status()

    return response.json()

In [7]:
data = search_track("Professional")

data

{'data': {'results': [{'isrc': 'GBAYE2200698',
    'title': 'Professional',
    'artist': 'Gabriels',
    'album': 'Angels & Queens (Deluxe)',
    'release_date': '2023-07-07T00:00:00.000Z'},
   {'isrc': 'USNA10319594',
    'title': 'Professional Daydreamer',
    'artist': 'Over the Rhine',
    'album': 'Ohio',
    'release_date': None},
   {'isrc': 'USAT29900495',
    'title': 'Professional Widow',
    'artist': 'Tori Amos',
    'album': 'Boys for Pele',
    'release_date': None},
   {'isrc': 'USAT20625633',
    'title': 'Professional Widow',
    'artist': 'Tori Amos',
    'album': 'Boys for Pele',
    'release_date': None},
   {'isrc': 'USRH11604040',
    'title': 'Professional Widow',
    'artist': 'Tori Amos',
    'album': 'Tales of a Librarian',
    'release_date': None},
   {'isrc': 'USAT20619231',
    'title': 'Professional Widow',
    'artist': 'Tori Amos',
    'album': 'Tales of a Librarian',
    'release_date': None},
   {'isrc': 'USRH11604343',
    'title': 'Professional Wid

In [8]:
import requests


def get_track_features(isrc):
    url = f"https://melodata1.p.rapidapi.com/tracks/{isrc}/features"

    headers = {
        "Content-Type": "application/json",
        "x-rapidapi-host": "melodata1.p.rapidapi.com",
        "x-rapidapi-key": "dbcd34ab68mshbdcca93581f9718p1c56b8jsn2fe95019b903"
    }

    response = requests.get(
        url,
        headers=headers
    )

    response.raise_for_status()

    return response.json()

In [9]:
features = get_track_features(
    "USUG11904206"
)

features

{'data': {'isrc': 'USUG11904206',
  'title': 'Blinding Lights',
  'artist': 'The Weeknd',
  'features': {'bpm': 170.8,
   'key': 'Fm',
   'key_confidence': 0.8761205,
   'energy': 0.7539831,
   'danceability': 0.4590787,
   'valence': None,
   'acousticness': None,
   'loudness': -8.5,
   'instrumentalness': None,
   'speechiness': 0.1168,
   'liveness': None,
   'time_signature': 4},
  'analysis_version': '1.2',
  'source': 'essentia'},
 'meta': {'quota': {'used': 0,
   'limit': None,
   'resets_at': '2026-09-13T14:04:40.981Z'},
  'request_id': 'req_8046ae065c1b'}}